# Marmousi acoustic point source

The equation, frequency, domain, boundary conditions and MHM Q3 configurations follow Section 6.4 of Chaumont-Frelet and Valentin (2020), DOI [10.1137/19M1255616](https://doi.org/10.1137/19M1255616). The material is a **declared primary-data crop**, not an identification of the historical coefficient arrays of Table 6.1.

The computational domain is 10,240 m by 2,560 m; the crop begins at primary coordinates (3395,515) m. Cell-centre samples form a 2048 by 512 grid with spacing 5 m. Density and velocity are converted to SI, with bulk modulus $\kappa=\rho c^2$. The unit point functional is at (5000,50) m, frequency 20 Hz, pressure zero on the top and first-order outgoing absorption on the other sides.


In [ ]:
from pathlib import Path
import hashlib
import json
import numpy as np
from IPython.display import Image, display
from examples.marmousi_fields import PixelCGField, reference_difference

ROOT = next(path for path in (Path.cwd(), *Path.cwd().parents) if (path / "pyproject.toml").exists())
SOURCE = ROOT / "examples/results/marmousi"
record = json.loads((SOURCE / "classical-convergence.json").read_text())
assert not record["source_changed_during_run"]
print(record["material"])


## A small exact replay check

The native field archive uses equispaced continuous Pk nodal coordinates on pixel-aligned SW–NE triangles. The example below evaluates a nonzero complex affine field and verifies an independent pressure integral. It does not require DOLFINx or the full primary SEG-Y files.


In [ ]:
x, z = np.meshgrid(np.arange(3), np.arange(4), indexing="ij")
fine = PixelCGField(1 + x + 2j*z, 1, (0, 2, 0, 3))
coarse = PixelCGField(np.ones((3,4), dtype=complex), 1, fine.bounds)
check = reference_difference(coarse, fine, 2*np.ones((2,3)), 3*np.ones((2,3)),
                             omega=2, gradient_cutout=(0,1,0,1))
assert np.isclose(check["pressure_difference"], np.sqrt(80), rtol=2e-13)
assert np.isclose(check["gradient_difference"], 5, rtol=2e-13)
check


## Physical and sampled reference increments

Global pressure L2 and the article's unweighted 513 by 129 sample norm are distinct. A two-dimensional Dirac source does not have a finite global H1 pressure norm. Gradient, acoustic flux and graph-norm comparisons therefore exclude the fixed square $(4975,5025)\times(25,75)$ m at every level. Pressure comparisons exclude nothing.

Relative increments use the finer reference field in the denominator. They measure the reference's own change, not a rigorous bound on its remaining error. Every native solve is checked against its original physical equations; this residual does not establish discretization accuracy.


In [ ]:
for row in record["references"]:
    archive = SOURCE / row["sample_archive"]
    assert hashlib.sha256(archive.read_bytes()).hexdigest() == row["sample_archive_sha256"]
    print("P", row["degree"], "complex DOFs", row["complex_dofs"])
    print("Native/replayed L2 difference:", row["native_pressure_norm_relative_difference"])
    if "increment" in row:
        print("Integrated increment:", row["increment_quadrature_check"])
        print("Unweighted sampled increment:", row["sample_relative_increment"])


## Material and complex fields

The overlays are the declared H=20 m and H=80 m MHM comparison partitions; they do not identify the displayed classical fields as MHM solutions. Real and imaginary pressure retain signed values. Their asinh display has linear width 1 and colorbar ticks in the original field values; it does not rescale the numerical data. The star marks the source.

An MHM value at a macro vertex can have four incident pressure values. Its acquisition stores all four without averaging; the article does not identify the incident sampling convention.

The maps use all native nodal pressure values, including the source node. Profiles at depth 500 m avoid the source and use linear amplitude axes.

For a source on a macroface, incident-angle allocation preserves the unit load. Here H=20 m and H=40 m assign one half to each adjacent macro; H=80 m assigns the full load to its single containing macro. This is a declared discrete convention, not an identified historical interface allocation.


In [ ]:
for name in ("material", "classical-fields-H20", "classical-fields-H80", "classical-profiles"):
    display(Image(filename=str(ROOT / f"docs/figures/marmousi/{name}.png")))


## MHM Q3 with H=20 m and a linear conormal trace

The MHM field uses the same data, source and boundary conditions as the P4 baseline. Common-overlay norms distinguish global pressure from derivative norms on the fixed cutout. All four incident conventions on the article's sampling grid remain separate; their 2.682–2.949% range brackets the printed 2.90%, without identifying its historical coefficient arrays or side convention.

The derivative differences are about 3%, compared with the P3→P4 reference flux increment of 0.0412%. Neither a refinement increment nor a small algebraic residual is a certified approximation-error bound.


In [ ]:
candidate_path = SOURCE / "mhm-H20-ell1.json"
candidate = json.loads(candidate_path.read_text())
comparison = json.loads((SOURCE / "mhm-H20-ell1-vs-classical-p4.json").read_text())
assert not candidate["source_changed_during_run"]
assert not comparison["source_changed_during_run"]
assert hashlib.sha256(candidate_path.read_bytes()).hexdigest() == comparison["candidate_record_sha256"]
with (SOURCE / candidate["archive"]).open("rb") as stream:
    assert hashlib.file_digest(stream, "sha256").hexdigest() == candidate["archive_sha256"]
print(comparison["norms"]["10"])
print(comparison["sampled_pressure"])
for name in ("mhm-H20-ell1-fields", "mhm-H20-ell1-profiles"):
    display(Image(filename=str(ROOT / f"docs/figures/marmousi/{name}.png")))
